In [ ]:
import csv
from columnTitles import column
import math
def createExperimentFile(descentArray):
    file = open("../../Project_Code/Python3Model/src/experiments/weather_files/experimentSetup.csv","w")
    csv_writer = csv.writer(file)

    csv_writer.writerow([
    "turbulence experiments",
    "Cognitive Delay",
    "reference_latitude",
    "reference_longitude",
    "reference_radial",
    "distance",
    "altitude_agl",
    "x_velocity",
    "y_velocity",
    "z_velocity",
    "p_rotation",
    "q_rotation",
    "r_rotation",
    "heading",
    "wind_altitude",
    "wind_direction",
    "wind_speed",
    "turbulence_level",
    "velocity_initial",
    "fuel_level"
    "average_descent_rate"
    ])

    csv_writer.writerow(
        [
    "null",
    "null",
    "null",
    "null",
    "null",
    "null",
    "null",
    "sim/flightmodel/position/local_vx",
    "sim/flightmodel/position/local_vy",
    "sim/flightmodel/position/local_vz",
    "sim/flightmodel/position/P",
    "sim/flightmodel/position/Q",
    "sim/flightmodel/position/R",
    "sim/flightmodel/position/q",
    "sim/weather/wind_altitude_msl_m[0]",
    "sim/weather/wind_direction_degt[0]",
    "sim/weather/wind_speed_kt[0]",
    "sim/weather/turbulence[0]",
    "sim/flightmodel/position/local_vz",
    "sim/flightmodel/weight/m_fuel"
    "null"
    ]
    )

    def distanceImageVelocity(descent):
        print(descent.columns.tolist())
        print(descent.columns.duplicated().any())
        print(descent.columns[descent.columns.duplicated()].tolist())
        print(type(descent[column.Altitude_AGL]))
        if column.Ground_Speed in descent.columns and column.Heading in descent.columns:

            index = 0
            for i in range(len(descent)):
                if descent[column.Altitude_AGL].iloc[i] < 5:
                    index = i
                    break

            totalX_nm = 0.0
            totalY_nm = 0.0

            for i in range(0, index):

                s1 = float(descent[column.Ground_Speed].iloc[i])
                s2 = float(descent[column.Ground_Speed].iloc[i+1])

                # if convert:
                #     s1 *= 1.94384
                #     s2 *= 1.94384

                avgS = (s1 + s2) / 2

                heading_deg = float(descent[column.Heading].iloc[i])
                heading_rad = math.radians(heading_deg)

                d_nm = avgS / 3600.0

                dx = d_nm * math.cos(heading_rad)
                dy = d_nm * math.sin(heading_rad)

                totalX_nm += dx
                totalY_nm += dy

            # Distance (magnitude)
            distance_nm = math.sqrt(totalX_nm**2 + totalY_nm**2)

            # Direction (bearing)
            bearing_rad = math.atan2(totalY_nm, totalX_nm)
            bearing_deg = (math.degrees(bearing_rad) + 360) % 360

            print("Distance:", distance_nm, "NM")
            print("Bearing:", bearing_deg, "deg")

            return [distance_nm, bearing_deg]

    def getAverageWindDirection(landing):
        total = 0
        for i in range(0,landing.__len__()):
            if column.Wind_Direction in landing:
                wd_rel = float(landing[column.Wind_Direction].iloc[i])
                hdg_mag = float(landing[column.Heading].iloc[i])
                mag_var = float(landing[column.Magnetic_Variation].iloc[i])  # east positive
                hdg_true = hdg_mag + mag_var
                wd_true = (wd_rel + hdg_true) % 360
                total += wd_true
        return total/landing.__len__()

    def getAverageWindSpeed(landing):
        total = 0
        for i in range(0,landing.__len__()):
            if column.Wind_Speed in landing:
                total+=float(landing[column.Wind_Speed].iloc[i])
        return total/landing.__len__()

    def getInitialVelocity(landing):
        if column.Ground_Speed in landing:
            return landing[column.Ground_Speed].iloc[0]
        
    def getInitialAltitudeAGL(landing):
        if column.Altitude_AGL in landing:
            return landing[column.Altitude_AGL].iloc[0]

    def getInitialHeading(landing):
        if column.Heading in landing:
            return landing[column.Heading].iloc[0]
        
    def getAverageDescentRate(landing):
        if column.Altitude_AGL in landing:
            return ((landing[column.Altitude_AGL].iloc[landing.__len__()-1] - landing[column.Altitude_AGL].iloc[0])/landing.__len__()) * 60


    count = 0
    for landing in descentArray:
        # for landing in descent:
            result = distanceImageVelocity(landing)
            cognitive_delay = 0.15
            lat = 39.875027
            long = -104.696482
            radial = 359
            distance = result[0]
            altitude_agl = getInitialAltitudeAGL(landing)-200
            x_velocity = 0
            y_velocity = 0
            z_velocity = 0
            p_rotation = 0
            q_rotation = 0
            r_rotation = 0
            heading = 179
            wind_altitude = 0
            wind_direction = getAverageWindDirection(landing)
            wind_speed = getAverageWindSpeed(landing) 
            turbulence_level = 0
            velocity_initial = getInitialVelocity(landing)
            fuel_level = 20
            average_descent_rate = getAverageDescentRate(landing)
            
            csv_writer.writerow([
                count,
                cognitive_delay,
                lat,
                long,
                radial,
                distance,
                altitude_agl,
                x_velocity,
                y_velocity,
                z_velocity,     
                p_rotation,
                q_rotation,
                r_rotation,
                heading,
                wind_altitude,
                wind_direction,
                wind_speed,
                turbulence_level,
                velocity_initial,
                fuel_level,
                average_descent_rate
            ])
            count+=1
    file.close()